# 🍌 SmartBanana — Training MobileNetV2 (Transfer Learning)
Notebook ini dibuat otomatis untuk mempermudah proses training menggunakan MobileNetV2.

### 🚀 CELL 1 — Import Library

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix

print('=' * 55)
print('✅ Semua library berhasil diimport!')
print(f'   TensorFlow versi : {tf.__version__}')
gpu = tf.config.list_physical_devices('GPU')
if gpu:
    print(f'   GPU              : ✅ Aktif ({gpu[0].name})')
else:
    print(f'   GPU              : ❌ Tidak aktif (aktifkan T4 GPU!)')
print('=' * 55)

### 🚀 CELL 2 — Mount Google Drive & Konfigurasi Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Sesuaikan path dataset kamu di Google Drive ──────────────────
DATASET_PATH  = '/content/drive/MyDrive/dataset_pisang'   # ← ubah jika perlu
MODEL_SAVE_DIR = '/content/drive/MyDrive/model_pisang'
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, 'cnn_kematangan_pisang.keras')
CONFIG_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, 'config.json')

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# ── Konfigurasi MobileNetV2 ───────────────────────────────────────
IMG_SIZE    = (224, 224)   # Optimal untuk MobileNetV2
BATCH_SIZE  = 32
EPOCHS_HEAD = 15           # Phase 1: hanya latih classifier
EPOCHS_FINE = 20           # Phase 2: fine-tune seluruh model
LR_HEAD     = 1e-3
LR_FINE     = 1e-5         # Harus sangat kecil saat fine-tuning

print(f'📂 Dataset path : {DATASET_PATH}')
print(f'💾 Model akan disimpan ke : {MODEL_SAVE_PATH}')
print(f'📐 Image size   : {IMG_SIZE}')

### 🚀 CELL 3 — Cek Dataset

In [ ]:
KELAS = sorted(os.listdir(DATASET_PATH))
print(f'\n📊 Kelas ditemukan: {KELAS}')
print(f'   Jumlah kelas  : {len(KELAS)}')
print()

total = 0
for k in KELAS:
    dir_k = os.path.join(DATASET_PATH, k)
    n = len([f for f in os.listdir(dir_k) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    total += n
    print(f'   📁 {k:<35} : {n} gambar')
print(f'\n   Total : {total} gambar')


# ─────────────────────────────────────────────────────────────────
# CELL 4 — Data Generator dengan Augmentasi
#           WAJIB pakai preprocess_input MobileNetV2 (bukan /255.0)
# ─────────────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ← WAJIB untuk MobileNetV2
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # ← WAJIB untuk MobileNetV2
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Simpan mapping kelas (penting agar urutan konsisten)
CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print(f'\n✅ Class mapping : {train_gen.class_indices}')
print(f'   CLASS_NAMES  : {CLASS_NAMES}')

### 🚀 CELL 5 — Bangun Model MobileNetV2

In [ ]:
def build_mobilenetv2_model(num_classes, img_size=(224, 224)):
    # 1. Base MobileNetV2 — bobot ImageNet, tanpa top layer
    base = MobileNetV2(
        input_shape=(*img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # Freeze semua layer base dulu (Phase 1)

    # 2. Tambah classifier custom
    inputs = keras.Input(shape=(*img_size, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base

model, base_model = build_mobilenetv2_model(NUM_CLASSES, IMG_SIZE)
model.summary()

total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'\n📊 Total params    : {total_params:,}')
print(f'   Trainable now  : {trainable_params:,}  (hanya head, base frozen)')

### 🚀 CELL 6 — Phase 1: Latih Classifier Head (Base Frozen)

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LR_HEAD),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb_early = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
)
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
)

print('🚀 Phase 1: Melatih Classifier Head (Base MobileNetV2 Frozen)...')
history_head = model.fit(
    train_gen,
    epochs=EPOCHS_HEAD,
    validation_data=val_gen,
    callbacks=[cb_early, cb_reduce_lr],
    verbose=1
)

### 🚀 CELL 7 — Phase 2: Fine-Tuning (Unfreeze sebagian base)

In [ ]:
# Unfreeze 50 layer terakhir dari MobileNetV2
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 50

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_now = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'🔓 Fine-tuning: {len(base_model.layers) - fine_tune_at} layer terakhir di-unfreeze')
print(f'   Trainable params sekarang: {trainable_now:,}')

# Compile ulang dengan LR sangat kecil
model.compile(
    optimizer=Adam(learning_rate=LR_FINE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cb_checkpoint = callbacks.ModelCheckpoint(
    MODEL_SAVE_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print('\n🚀 Phase 2: Fine-Tuning MobileNetV2...')
history_fine = model.fit(
    train_gen,
    epochs=EPOCHS_FINE,
    validation_data=val_gen,
    callbacks=[cb_early, cb_reduce_lr, cb_checkpoint],
    verbose=1
)

### 🚀 CELL 8 — Evaluasi & Confusion Matrix

In [ ]:
print('\n📊 Evaluasi pada Validation Set...')
val_gen.reset()
loss, acc = model.evaluate(val_gen, verbose=0)
print(f'   Val Loss     : {loss:.4f}')
print(f'   Val Accuracy : {acc:.4f} ({acc*100:.2f}%)')

# Prediksi semua validation data
val_gen.reset()
preds = model.predict(val_gen, verbose=0)
y_pred = np.argmax(preds, axis=1)
y_true = val_gen.classes

# Classification Report
print('\n' + '=' * 55)
print('📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — MobileNetV2', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### 🚀 CELL 9 — Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gabungkan history
all_acc     = history_head.history['accuracy']     + history_fine.history['accuracy']
all_val_acc = history_head.history['val_accuracy'] + history_fine.history['val_accuracy']
all_loss    = history_head.history['loss']         + history_fine.history['loss']
all_val_loss= history_head.history['val_loss']     + history_fine.history['val_loss']

ep_head = len(history_head.history['accuracy'])

# Accuracy
axes[0].plot(all_acc,     label='Train Accuracy', color='royalblue')
axes[0].plot(all_val_acc, label='Val Accuracy',   color='orange')
axes[0].axvline(ep_head, color='gray', linestyle='--', label='Fine-tune start')
axes[0].set_title('Akurasi Training', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(all_loss,     label='Train Loss', color='royalblue')
axes[1].plot(all_val_loss, label='Val Loss',   color='orange')
axes[1].axvline(ep_head, color='gray', linestyle='--', label='Fine-tune start')
axes[1].set_title('Loss Training', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('🍌 MobileNetV2 Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 🚀 CELL 10 — Simpan Model + Update config.json

In [ ]:
# Simpan model jika belum tersimpan oleh callback (simpan manual sekali lagi)
model.save(MODEL_SAVE_PATH)
print(f'✅ Model disimpan ke: {MODEL_SAVE_PATH}')

# Update config.json — PENTING: ubah img_size ke 224
config = {
    "img_size": list(IMG_SIZE),          # [224, 224]
    "class_names": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "val_accuracy": float(acc),
    "model_type": "mobilenetv2",
    "preprocessing": "mobilenet_v2"      # ← penanda untuk HF
}

with open(CONFIG_SAVE_PATH, 'w') as f:
    json.dump(config, f, indent=2)

print(f'✅ config.json diperbarui:')
print(json.dumps(config, indent=2))

### 🚀 CELL 11 — Fungsi Prediksi (Inference) — WAJIB pakai preprocess_input

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

LABEL_DISPLAY = {
    'banana unripe':     '🟢 Unripe (Mentah)',
    'banana semi-ripe':  '🟡 Semi-Ripe (Setengah Matang)',
    'banana fully-ripe': '🟠 Fully-Ripe (Matang)',
}
WARNA_KELAS = ['#f97316', '#eab308', '#22c55e']


def prediksi_kematangan_pisang(path_gambar):
    """Prediksi tingkat kematangan pisang — MobileNetV2 version."""

    # 1. Load & Preprocess — WAJIB preprocess_input, BUKAN /255.0
    img = Image.open(path_gambar).convert('RGB').resize(IMG_SIZE)
    arr = np.array(img, dtype=np.float32)
    arr = preprocess_input(arr)            # ← range [-1, 1]
    arr = np.expand_dims(arr, axis=0)      # (1, 224, 224, 3)

    # 2. Prediksi
    probs     = model.predict(arr, verbose=0)[0]
    idx       = np.argmax(probs)
    kelas     = CLASS_NAMES[idx]
    keyakinan = probs[idx] * 100
    label     = LABEL_DISPLAY.get(kelas, kelas)

    # 3. Visualisasi
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(img)
    axes[0].set_title(f'Hasil: {label}\nKeyakinan: {keyakinan:.1f}%',
                      fontsize=13, fontweight='bold')
    axes[0].axis('off')

    label_list = [LABEL_DISPLAY.get(c, c) for c in CLASS_NAMES]
    warna_bar  = [WARNA_KELAS[i] if i == idx else '#cbd5e1' for i in range(NUM_CLASSES)]
    bars = axes[1].barh(label_list, probs * 100, color=warna_bar,
                        edgecolor='black', linewidth=0.6)
    for bar, p in zip(bars, probs):
        axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                     f'{p*100:.1f}%', va='center', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Probabilitas (%)')
    axes[1].set_title('Distribusi Probabilitas', fontsize=12, fontweight='bold')
    axes[1].set_xlim([0, 120])
    axes[1].grid(True, axis='x', alpha=0.3)

    plt.suptitle('🍌 Hasil Prediksi Kematangan Pisang (MobileNetV2)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'\n   Pisang ini tergolong: {label}')
    print(f'   Tingkat keyakinan   : {keyakinan:.2f}%')

    return kelas, keyakinan

### 🚀 CELL 12 — Test Prediksi Gambar Sendiri

In [ ]:
from google.colab import files
print('📤 Pilih gambar pisang dari komputer Anda...')
uploaded = files.upload()

for nama_file in uploaded.keys():
    path_upload = f'/content/{nama_file}'
    print(f'\n🖼️  Memprediksi: {nama_file}')
    prediksi_kematangan_pisang(path_upload)